In [1]:
# 01_data_loading.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 01. Data Loading
#
# Load and inspect three credit-domain datasets.
# - German Credit (UCI) — sensitive: sex, age
# - Taiwan Credit Card Default (UCI) — sensitive: sex, education
# - Credit Card Approval Prediction (Kaggle) — sensitive: sex
#   Targets derived from credit_record.csv at two delinquency thresholds (30dpd, 60dpd).

# %%
# Imports and path setup
import os
import urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Paths are relative to the notebooks/ folder
DATA_DIR = os.path.join("..", "data")
RESULTS_DIR = os.path.join("..", "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

os.makedirs(DATA_DIR, exist_ok=True)

# %%
# Download UCI raw files if missing (German Credit, Taiwan Credit Card Default).
# Credit Card Approval files (application_record.csv, credit_record.csv) must be
# placed in data/ manually (Kaggle dataset).
german_file = os.path.join(DATA_DIR, "german.data")
if not os.path.exists(german_file):
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
        german_file,
    )

taiwan_file = os.path.join(DATA_DIR, "default of credit card clients.xls")
if not os.path.exists(taiwan_file):
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00350/default%20of%20credit%20card%20clients.xls",
        taiwan_file,
    )

# %%
# Load German Credit (Statlog) — UCI
# Raw german.data is whitespace-separated, no header, coded categorical values
# https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data
german_cols = [
    "status_checking", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status_sex", "other_debtors",
    "residence_since", "property", "age", "other_installment", "housing",
    "existing_credits", "job", "num_dependents", "telephone", "foreign_worker", "target"
]

df_german = pd.read_csv(german_file, sep=r"\s+", header=None, names=german_cols)

# Recode target: original 1 = good, 2 = bad -> 1 = bad (default), 0 = good
df_german["target"] = (df_german["target"] == 2).astype(int)

# Derive sensitive attributes
# personal_status_sex codes:
# A91: male divorced, A92: female, A93: male single, A94: male married, A95: female
sex_map = {"A91": "male", "A92": "female", "A93": "male", "A94": "male", "A95": "female"}
df_german["sex"] = df_german["personal_status_sex"].map(sex_map)
df_german["age_group"] = np.where(df_german["age"] >= 25, "old", "young")

print("German Credit shape:", df_german.shape)
df_german.head()

# %%
# Load Taiwan Credit Card Default — UCI
# Raw file is .xls with a title row, so use header=1 (requires xlrd >= 2.0.1)
# https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
df_taiwan = pd.read_excel(taiwan_file, header=1)

df_taiwan = df_taiwan.rename(columns={"default payment next month": "target"})
if "ID" in df_taiwan.columns:
    df_taiwan = df_taiwan.drop(columns=["ID"])

# Sensitive attributes: SEX (1=male, 2=female), EDUCATION (1=grad, 2=univ, 3=hs, else other)
df_taiwan["sex"] = df_taiwan["SEX"].map({1: "male", 2: "female"})
df_taiwan["education"] = df_taiwan["EDUCATION"].map({
    1: "graduate", 2: "university", 3: "high_school",
    0: "others", 4: "others", 5: "others", 6: "others",
})

print("Taiwan shape:", df_taiwan.shape)
print(df_taiwan["target"].value_counts(normalize=True).round(3))
df_taiwan.head()

# %%
# Build targets from Credit Card Approval Prediction — Kaggle
# Two files: application_record.csv (features), credit_record.csv (monthly STATUS).
# STATUS codes: 0=1-29d, 1=30-59d, 2=60-89d, 3=90-119d, 4=120-149d, 5=150+d/writeoff,
#               C=paid off, X=no loan.
# Two labels are built at different delinquency thresholds:
#   target_30dpd : bad if any month reaches 30+ days past due  (STATUS in {1,2,3,4,5})
#   target_60dpd : bad if any month reaches 60+ days past due  (STATUS in {2,3,4,5})
PRIMARY_TARGET = "target_30dpd"  # which threshold column is used as the default "target"

df_app = pd.read_csv(os.path.join(DATA_DIR, "application_record.csv"))
df_credit = pd.read_csv(os.path.join(DATA_DIR, "credit_record.csv"))

# Map STATUS to numeric past-due severity; C (paid off) and X (no loan) -> not in default
status_num = df_credit["STATUS"].replace({"C": -1, "X": -1}).astype(int)
df_credit = df_credit.assign(status_num=status_num)

# Per-client worst (maximum) past-due severity across all months
worst = df_credit.groupby("ID")["status_num"].max()

# Two threshold-based labels
labels = pd.DataFrame({
    "target_30dpd": (worst >= 1).astype(int),
    "target_60dpd": (worst >= 2).astype(int),
})

# Merge features with labels (keep applicants that have credit history)
df_home = df_app.merge(labels, on="ID", how="inner")

# Set the primary target alias used by downstream notebooks
df_home["target"] = df_home[PRIMARY_TARGET]

# Sensitive attribute: sex
df_home["sex"] = df_home["CODE_GENDER"].map({"M": "male", "F": "female"})
df_home = df_home[df_home["sex"].notna()].copy()

print("Credit Card Approval shape:", df_home.shape)
print("Primary target:", PRIMARY_TARGET)
print("target_30dpd pos_rate:", round(df_home["target_30dpd"].mean(), 3))
print("target_60dpd pos_rate:", round(df_home["target_60dpd"].mean(), 3))
df_home[["ID", "target", "target_30dpd", "target_60dpd", "sex", "AMT_INCOME_TOTAL"]].head()

# %%
# Build summary table across the three datasets
summary = pd.DataFrame([
    {"dataset": "German Credit", "n_rows": len(df_german), "n_cols": df_german.shape[1],
     "pos_rate": round(df_german["target"].mean(), 3), "sensitive": "sex, age_group"},
    {"dataset": "Taiwan CC Default", "n_rows": len(df_taiwan), "n_cols": df_taiwan.shape[1],
     "pos_rate": round(df_taiwan["target"].mean(), 3), "sensitive": "sex, education"},
    {"dataset": "Credit Card Approval (30dpd)", "n_rows": len(df_home), "n_cols": df_home.shape[1],
     "pos_rate": round(df_home["target_30dpd"].mean(), 3), "sensitive": "sex"},
    {"dataset": "Credit Card Approval (60dpd)", "n_rows": len(df_home), "n_cols": df_home.shape[1],
     "pos_rate": round(df_home["target_60dpd"].mean(), 3), "sensitive": "sex"},
])
print(summary.to_string(index=False))

# Save summary table
os.makedirs(TAB_DIR, exist_ok=True)
summary.to_csv(os.path.join(TAB_DIR, "01_dataset_summary.csv"), index=False)

# %%
# Persist intermediate frames for reuse in the next notebook
os.makedirs(os.path.join(DATA_DIR, "processed"), exist_ok=True)
df_german.to_parquet(os.path.join(DATA_DIR, "processed", "german.parquet"))
df_taiwan.to_parquet(os.path.join(DATA_DIR, "processed", "taiwan.parquet"))
df_home.to_parquet(os.path.join(DATA_DIR, "processed", "credit_card_approval.parquet"))

German Credit shape: (1000, 23)
Taiwan shape: (30000, 26)
target
0    0.779
1    0.221
Name: proportion, dtype: float64
Credit Card Approval shape: (36457, 22)
Primary target: target_30dpd
target_30dpd pos_rate: 0.118
target_60dpd pos_rate: 0.017
                     dataset  n_rows  n_cols  pos_rate      sensitive
               German Credit    1000      23     0.300 sex, age_group
           Taiwan CC Default   30000      26     0.221 sex, education
Credit Card Approval (30dpd)   36457      22     0.118            sex
Credit Card Approval (60dpd)   36457      22     0.017            sex
